# Lab 4 — Numerical Methods, and a Fast RBC Model

**ECON 282E · Session 4 · October 15, 2026**

Session 3 made six crude choices and measured what each one cost. This lab builds the
replacement for each, and measures what it bought.

| You build | It repairs |
|---|---|
| bisection, Newton, secant, Brent | `brentq` as a sealed black box |
| golden section | the grid search for the maximum |
| shape-preserving interpolation | linear interpolation only |
| Gauss--Hermite quadrature | the expectation as a crude sum |
| Tauchen and Rouwenhorst | the transition matrix written by hand |
| Howard and EGM | 1,352 sweeps and 2.4 billion comparisons |

**Prerequisite:** L03. Everything here is NumPy and SciPy; nothing needs the internet.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline, PchipInterpolator
from scipy.stats import norm

np.set_printoptions(precision=6, suppress=True)

---
## Part 1 — Root-finding, and how methods fail

A market-clearing condition: $Q^d(p) = 100p^{-1/2}$ against $Q^s(p) = e^{p/2}-1$.

In [ ]:
demand   = lambda p: 100.0 * p**-0.5
supply   = lambda p: np.exp(0.5*p) - 1.0
excess   = lambda p: demand(p) - supply(p)
d_excess = lambda p: -50.0*p**-1.5 - 0.5*np.exp(0.5*p)

p_star = brentq(excess, 0.1, 20.0, xtol=1e-15)
print(f"p* = {p_star:.10f},  residual = {excess(p_star):.2e}")

In [ ]:
def bisection(f, a, b, tol=1e-12, maxit=200):
    hist = []
    for n in range(maxit):
        m = 0.5*(a+b)
        if f(a)*f(m) <= 0: b = m
        else:              a = m
        hist.append(0.5*(a+b))
        if b-a < tol: break
    return np.array(hist)

def newton(f, fp, x0, tol=1e-12, maxit=200):
    hist, x = [], x0
    for n in range(maxit):
        x = x - f(x)/fp(x)
        hist.append(x)
        if abs(f(x)) < tol: break
    return np.array(hist)

def secant(f, x0, x1, tol=1e-12, maxit=200):
    hist = []
    for n in range(maxit):
        f0, f1 = f(x0), f(x1)
        if f1 == f0: break
        x0, x1 = x1, x1 - f1*(x1-x0)/(f1-f0)
        hist.append(x1)
        if abs(f(x1)) < tol: break
    return np.array(hist)

hists = {"bisection": bisection(excess, 0.1, 20.0),
         "newton":    newton(excess, d_excess, 10.0),
         "secant":    secant(excess, 6.0, 9.0)}

for name, h in hists.items():
    e = np.abs(h - p_star)
    e = e[e > 1e-14]
    if name == "bisection":
        # the bisection error is not monotone, so the 3-point order formula is
        # meaningless here.  Fit the LINEAR rate instead: e_n ~ C * r^n.
        r = np.exp(np.polyfit(np.arange(e.size), np.log(e), 1)[0])
        print(f"{name:10s} {len(h):3d} iterations, error ratio per step {r:.3f} (theory 0.5)")
    else:
        order = np.log(e[-1]/e[-2]) / np.log(e[-2]/e[-3])
        print(f"{name:10s} {len(h):3d} iterations, observed order {order:.2f}")

In [ ]:
plt.figure(figsize=(7, 4))
for name, h in hists.items():
    plt.plot(np.arange(1, len(h)+1), np.log10(np.maximum(np.abs(h-p_star), 1e-16)),
             marker='o', ms=2.5, label=name)
plt.xlabel("iteration"); plt.ylabel(r"$\log_{10}|x_n - x^*|$")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

The three slopes are the three convergence orders: linear, superlinear (golden ratio) and
quadratic. Read them off the plot and check them against the printed numbers.

**Exercise 1.** Now a demand/supply pair with **no** equilibrium:
$Q^d = 100-2p+10\sqrt p$, $Q^s = -20+3p-0.1p^2$. Run all three methods on it. Which ones
tell you something is wrong, and which ones return a number? What is the residual there?

In [ ]:
bad = lambda p: (100 - 2*p + 10*np.sqrt(np.maximum(p, 1e-12))) - (-20 + 3*p - 0.1*p**2)
pg = np.linspace(0.01, 200, 5000)
print(f"minimum excess demand over the grid: {bad(pg).min():.2f} at p = {pg[bad(pg).argmin()]:.2f}")
try:
    brentq(bad, 1, 50)
except ValueError as e:
    print("brentq refuses:", e)
h = newton(bad, lambda p: (bad(p+1e-7)-bad(p-1e-7))/2e-7, 20.0, maxit=50)
print(f"Newton returns p = {h[-1]:.2f}, where excess demand is {bad(h[-1]):.1f}")

---
## Part 2 — Optimization on an interval

Golden section: shrink a bracket by the ratio that lets you recycle one function value.

In [ ]:
PHI = (np.sqrt(5.0) - 1.0) / 2.0

def golden_section(f, a, b, tol=1e-8):
    n = 0
    c, d = b - PHI*(b-a), a + PHI*(b-a)
    fc, fd = f(c), f(d); n += 2
    while b - a > tol:
        if fc > fd: b, d, fd = d, c, fc; c = b - PHI*(b-a); fc = f(c)
        else:       a, c, fc = c, d, fd; d = a + PHI*(b-a); fd = f(d)
        n += 1
    return 0.5*(a+b), n

f = lambda x: -(x - 2.7)**2 + 3.0
for tol in (1e-4, 1e-6, 1e-8):
    x, n = golden_section(f, 0.0, 10.0, tol)
    print(f"tol {tol:.0e}: golden section {n:3d} evaluations   "
          f"grid search would need {int(np.ceil(10/tol)):,}")

Each extra decimal digit costs golden section **five more evaluations** and costs the grid a
factor of ten. And the cost does not depend on the state grid at all.

**Exercise 2.** Verify the ratio. Solve $	au = (1-	au)^2$ and check that $1-	au$ is what
`PHI` holds. Why is that the unique ratio that permits recycling?

---
## Part 3 — Interpolation: accuracy is not the criterion

A policy with a binding constraint, $g(k) = \min(k, 4)$.

In [ ]:
k_nodes = np.linspace(0, 10, 21)
g_nodes = np.minimum(k_nodes, 4.0)
kf = np.linspace(0, 10, 2000)
truth = np.minimum(kf, 4.0)

fits = {"linear":       np.interp(kf, k_nodes, g_nodes),
        "cubic spline": CubicSpline(k_nodes, g_nodes, bc_type='natural')(kf),
        "pchip":        PchipInterpolator(k_nodes, g_nodes)(kf)}

for name, y in fits.items():
    print(f"{name:13s} max error {np.max(np.abs(y-truth)):.4f}   "
          f"overshoot above the cap {max(y.max()-4.0, 0.0):.4f}   "
          f"monotone {bool(np.all(np.diff(y) >= -1e-12))}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(kf, truth, 'k--', lw=1.2, label='truth: min(k, 4)')
for name, y in fits.items():
    plt.plot(kf, y, lw=1.2, label=name)
plt.axhline(4.0, color='grey', ls=':', lw=1)
plt.xlim(2.5, 7); plt.ylim(3.2, 4.3)
plt.xlabel("$k$"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

The cubic spline is the most accurate scheme on smooth data and it is the wrong one here: it
puts the policy **above a cap the model says cannot be exceeded**. Feed that back into a
Bellman loop and the inner maximization is no longer unimodal.

**Exercise 3.** Check the claim the lecture corrected: interpolate $\ln k$ on 12 nodes and
verify that the *linear* interpolant is concave (all second differences $\le 0$). Then check
the cubic spline on `min(k,4)` and show that it is not.

### Part 3b — Smolyak: the nodes you keep when $d$ gets large

One dimension at a time is easy. Two continuous states cost $n^2$ points, ten cost $n^{10}$.
Smolyak's sparse grid keeps only the index combinations whose *total* resolution is modest.

In [ ]:
from itertools import product

def nodes_1d(i):
    """G^i: Chebyshev extrema, m_1 = 1 and m_i = 2^(i-1)+1. Nested by construction."""
    if i == 1:
        return np.array([0.0])
    m = 2**(i-1) + 1
    return -np.cos(np.pi*np.arange(m)/(m-1))

def index_sets(d, mu):
    """All (i_1..i_d) with i_j >= 1 and sum i_j <= d + mu."""
    if d == 1:
        for a in range(mu+1):
            yield (a+1,)
        return
    for a in range(mu+1):
        for rest in index_sets(d-1, mu-a):
            yield (a+1,) + rest

def smolyak_grid(d, mu):
    pts = set()
    for idx in index_sets(d, mu):
        for combo in product(*[nodes_1d(i) for i in idx]):
            pts.add(tuple(round(float(c), 12) for c in combo))
    return sorted(pts)

# nestedness is the property the whole construction rests on
for i in (1, 2, 3):
    a, b = set(np.round(nodes_1d(i), 10)), set(np.round(nodes_1d(i+1), 10))
    print(f"G^{i} ({len(a)} nodes) subset of G^{i+1} ({len(b)} nodes): {a <= b}")

In [ ]:
print(f"{'d':>3} {'mu=1':>6} {'mu=2':>6} {'mu=3':>7} {'5^d':>12} {'9^d':>14}")
for d in (2, 3, 4, 5, 10, 12):
    row = [len(smolyak_grid(d, mu)) if (d <= 5 or mu <= 2) else None for mu in (1, 2, 3)]
    cells = "".join(f"{v:>7,}" if v is not None else f"{'--':>7}" for v in row)
    print(f"{d:>3}{cells} {5**d:>12,} {9**d:>14,}")

# Malin, Krueger & Kubler (2011) print this column; check against it
print()
for d, n in {2: 13, 4: 41, 5: 61, 12: 313}.items():
    print(f"  d={d:>2}, mu=2: ours {len(smolyak_grid(d,2)):>4}   "
          f"paper {n:>4}   {'match' if len(smolyak_grid(d,2))==n else 'MISMATCH'}")

In [ ]:
sp = np.array(smolyak_grid(2, 2))
g5 = nodes_1d(3)
tensor = np.array([(a, b) for a in g5 for b in g5])

plt.figure(figsize=(4.4, 4.4))
plt.scatter(tensor[:,0], tensor[:,1], s=42, facecolors='none', edgecolors='grey',
            label=f'tensor 5x5 ({len(tensor)})')
plt.scatter(sp[:,0], sp[:,1], s=30, color='C3', label=f'Smolyak mu=2 ({len(sp)})')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
plt.xticks([-1,0,1]); plt.yticks([-1,0,1]); plt.gca().set_aspect('equal')
plt.tight_layout(); plt.show()

The sparse grid keeps the **axes and the corners** and throws away the interior cross-terms.
At $d=10$ that is 221 points against 9,765,625 — a factor of 44,000.

**Exercise 3b.** Interpolate $f(x,y)=\exp(-(x^2+y^2))$ on the Smolyak $\mu=3$ grid and on
a tensor grid with a comparable number of points, and compare maximum errors on 5,000 random
test points. Which wins, and does the answer change if you replace $f$ with something whose
cross-derivative is large?

---
## Part 4 — Quadrature

$\mathbb{E}[e^z]$ for $z \sim N(0,1)$, whose exact value is $e^{1/2}$.

In [ ]:
exact = np.exp(0.5)
g = lambda z: np.exp(z)

print(f"{'n':>4}  {'trapezoid':>12}  {'Gauss-Hermite':>14}  {'Monte Carlo':>12}")
rng = np.random.default_rng(0)
for n in (3, 5, 9, 17):
    a, b = -8.0, 8.0
    x = np.linspace(a, b, n)
    w = np.full(n, (b-a)/(n-1)); w[0] *= 0.5; w[-1] *= 0.5
    trap = np.sum(w * g(x) * norm.pdf(x))
    xh, wh = np.polynomial.hermite_e.hermegauss(n)
    gh = np.sum(wh * g(xh)) / np.sqrt(2*np.pi)
    mc = g(rng.standard_normal(n)).mean()
    print(f"{n:4d}  {abs(trap-exact):12.2e}  {abs(gh-exact):14.2e}  {abs(mc-exact):12.2e}")

Gauss--Hermite knows the weight is a normal density, so it places its nodes where the
probability is. **It is exact at 17 nodes** — a million Monte Carlo draws are not.

**Exercise 4.** Add Simpson's rule to the table. It is a higher-order rule than the trapezoid
rule. Is it more accurate here? Explain what you find.

---
## Part 5 — Where $\Pi$ comes from

Two ways to turn $\ln z' = ho \ln z + \sigmaarepsilon'$ into a finite Markov chain.

In [ ]:
def tauchen(n, rho, sigma, m=3.0):
    sz = sigma / np.sqrt(1 - rho**2)
    y = np.linspace(-m*sz, m*sz, n)
    step = y[1] - y[0]
    P = np.empty((n, n))
    for i in range(n):
        P[i, 0]  = norm.cdf((y[0]  - rho*y[i] + step/2) / sigma)
        P[i, -1] = 1 - norm.cdf((y[-1] - rho*y[i] - step/2) / sigma)
        for j in range(1, n-1):
            P[i, j] = (norm.cdf((y[j] - rho*y[i] + step/2)/sigma)
                       - norm.cdf((y[j] - rho*y[i] - step/2)/sigma))
    return y, P

def rouwenhorst(n, rho, sigma):
    p = (1 + rho) / 2
    P = np.array([[p, 1-p], [1-p, p]])
    for k in range(3, n+1):
        Z = np.zeros((k, k))
        Z[:-1, :-1] += p*P;     Z[:-1, 1:] += (1-p)*P
        Z[1:,  :-1] += (1-p)*P; Z[1:,  1:] += p*P
        Z[1:-1, :] /= 2.0
        P = Z
    sz = sigma / np.sqrt(1 - rho**2)
    psi = sz * np.sqrt(n - 1)
    return np.linspace(-psi, psi, n), P

def chain_moments(y, P):
    w, v = np.linalg.eig(P.T)
    pi = np.real(v[:, np.argmin(np.abs(w - 1))]); pi /= pi.sum()
    mu = pi @ y
    var = pi @ (y - mu)**2
    cov = sum(pi[a]*P[a, b]*(y[a]-mu)*(y[b]-mu) for a in range(len(y)) for b in range(len(y)))
    return np.sqrt(var), cov/var

In [ ]:
sigma = 0.007
print(f"{'rho':>5} {'n':>3} | {'Tauchen rho':>12} {'sd err %':>9} | "
      f"{'Rouwen. rho':>12} {'sd err %':>9}")
for rho in (0.90, 0.95, 0.99):
    target_sd = sigma / np.sqrt(1 - rho**2)
    for n in (5, 9, 15):
        sdt, rt = chain_moments(*tauchen(n, rho, sigma))
        sdr, rr = chain_moments(*rouwenhorst(n, rho, sigma))
        print(f"{rho:5.2f} {n:3d} | {rt:12.4f} {100*abs(sdt/target_sd-1):9.1f} | "
              f"{rr:12.6f} {100*abs(sdr/target_sd-1):9.1e}")

Look at the $ho = 0.99$, $n = 5$ row: the Tauchen chain reports a persistence of **1.0000**
— a unit root — and an unconditional standard deviation more than a third too small. At
$n=15$ it is still about a fifth too small. Rouwenhorst is exact to machine precision
everywhere, because it matches $ho$ and $\sigma_z$ *by construction*.

**Use Rouwenhorst**, and report the chain's implied moments next to the targets. Homework 1
Part B asks for exactly this table.

---
## Part 6 — Putting it together: a fast RBC solve

Quarterly calibration: $lpha=0.33$, $eta=0.99$, $\delta=0.025$, $ho=0.95$,
$\sigma=0.007$.

In [ ]:
class RBC:
    def __init__(self, n_k=300, n_z=7, width=0.5):
        self.alpha, self.beta, self.delta = 0.33, 0.99, 0.025
        logz, self.Pi = rouwenhorst(n_z, 0.95, 0.007)
        self.z = np.exp(logz)
        self.kss = ((1/self.beta - 1 + self.delta)/self.alpha)**(1/(self.alpha-1))
        self.k = np.linspace((1-width)*self.kss, (1+width)*self.kss, n_k)
        self.n_k, self.n_z = n_k, n_z
        self.y = self.z[None,:]*self.k[:,None]**self.alpha + (1-self.delta)*self.k[:,None]
        C = self.y[:,:,None] - self.k[None,None,:]
        self.U = np.full_like(C, -1e10)
        np.log(C, out=self.U, where=C > 0)

m = RBC()                      # n_k = 300 here; the lecture used 500, so times differ
print(f"k_ss = {m.kss:.4f},  K/Y = {m.kss/m.kss**m.alpha:.2f} (quarterly)")

In [ ]:
def vfi(model, n_howard=0, tol=1e-6, maxit=20000):
    V = np.zeros((model.n_k, model.n_z))
    ik, iz = np.meshgrid(np.arange(model.n_k), np.arange(model.n_z), indexing='ij')
    t0, outer = time.perf_counter(), 0
    while outer < maxit:
        EV = V @ model.Pi.T
        M = model.U + model.beta * EV.T[None, :, :]
        Vn, pol = M.max(axis=2), M.argmax(axis=2)
        d = np.max(np.abs(Vn - V)); V = Vn; outer += 1
        for _ in range(n_howard):                    # policy evaluation only
            EV = V @ model.Pi.T
            V = model.U[ik, iz, pol] + model.beta * EV[pol, iz]
        if d < tol: break
    return V, model.k[pol], outer, time.perf_counter() - t0

for n_h in (0, 10, 20, 50):
    _, g, it, el = vfi(m, n_howard=n_h)
    label = "plain VFI" if n_h == 0 else f"Howard n_H={n_h}"
    print(f"{label:16s} {it:5d} policy updates   {el:6.2f} s")

Howard's policy improvement replaces most of the maximizations with a lookup. The answer is
identical; only the route to it changes.

Now the endogenous grid method — **on cash on hand**, which is what removes the root-finding.
Putting the grid on $k$ instead reintroduces a solve at every node, and the method loses the
speed race it exists to win.

In [ ]:
def egm(model, tol=1e-10, maxit=5000):
    kp = model.k.copy()                     # end-of-period assets
    mg = np.linspace(model.y.min(), model.y.max(), model.n_k)   # cash on hand
    c = 0.3 * mg[:, None] * np.ones((1, model.n_z))
    t0, it = time.perf_counter(), 0
    while it < maxit:
        mp = model.z[None,:]*kp[:,None]**model.alpha + (1-model.delta)*kp[:,None]
        R  = model.alpha*model.z[None,:]*kp[:,None]**(model.alpha-1) + (1-model.delta)
        cp = np.empty_like(mp)
        for l in range(model.n_z):
            cp[:, l] = np.interp(mp[:, l], mg, c[:, l])
        rhs   = model.beta * ((R/cp) @ model.Pi.T)
        c_end = 1.0/rhs                      # invert marginal utility
        m_end = c_end + kp[:, None]          # <- closed form: NO root-finding
        c_new = np.empty_like(c)
        for j in range(model.n_z):
            c_new[:, j] = np.interp(mg, m_end[:, j], c_end[:, j])
            tight = mg < m_end[0, j]
            c_new[tight, j] = mg[tight] - kp[0]
        d = np.max(np.abs(c_new - c)); c = c_new; it += 1
        if d < tol: break
    return mg, c, it, time.perf_counter() - t0

mg, cpol, it, el = egm(m)
print(f"EGM: {it} iterations, {el:.2f} s, 0 root solves")

**Exercise 5.** Compute off-grid Euler residuals for the plain VFI policy and for the EGM
policy and compare. (L03 Part 6 has the residual function; adapt it to this calibration.)
You should find four to five decades between them.

**Exercise 6.** Time the monotone variant: exploit $g(k,z)$ increasing to start each state's
search where the previous one stopped. Count comparisons *and* wall time. They do not move
in the same direction — explain why.

---
## Part 7 — The worked example of deck 4.B5, end to end

Everything above was one tool at a time. This part is the whole toolkit in one solver, and it
is the code behind **deck 4.B5**. Four substitutions relative to the crude Session 3 solver:

| 3.A4 | here |
|---|---|
| $\Pi$ written by hand | **Rouwenhorst** (Part 5) |
| no interpolation | **linear**, because it keeps $\hat v$ concave (Part 3) |
| grid search for the max | **golden section** (Part 2) |
| a full sweep every iteration | **Howard**, $n_H = 20$ (Part 6) |

Note the order of the argument, because it is the point of the deck: linear interpolation
keeps $\hat v$ concave → the objective is unimodal → golden section is *licensed*. Swap in a
cubic spline and the first link breaks, so the third one does too.

In [ ]:
def rhs(kp, yj, EVj, model, kg):
    """Bellman right-hand side, evaluated OFF the grid."""
    c = np.maximum(yj - kp, 1e-12)             # clamp acts as a feasibility penalty
    return np.log(c) + model.beta*np.interp(kp, kg, EVj)

def vfi_continuous(model, n_howard=20, tol=1e-6, maxit=20000, n_gs=40):
    kg = model.k
    V  = np.zeros((model.n_k, model.n_z)); Vn = np.empty_like(V)
    g  = np.empty_like(V)
    t0, outer, nev = time.perf_counter(), 0, 0
    while outer < maxit:
        EV = V @ model.Pi.T                    # the whole expectation, one matmul
        for j in range(model.n_z):
            yj, EVj = model.y[:, j], EV[:, j]
            a = np.full(model.n_k, kg[0])
            b = np.minimum(yj - 1e-8, kg[-1])
            c_, d_ = b - PHI*(b-a), a + PHI*(b-a)
            fc = rhs(c_, yj, EVj, model, kg); fd = rhs(d_, yj, EVj, model, kg)
            nev += 2*model.n_k
            for _ in range(n_gs):              # golden section, vectorised over all k
                L = fc > fd
                b = np.where(L, d_, b); a = np.where(L, a, c_)
                c_, d_ = b - PHI*(b-a), a + PHI*(b-a)
                fc = rhs(c_, yj, EVj, model, kg); fd = rhs(d_, yj, EVj, model, kg)
                nev += 2*model.n_k
            g[:, j]  = 0.5*(a + b)
            Vn[:, j] = rhs(g[:, j], yj, EVj, model, kg)
        d = np.max(np.abs(Vn - V))
        V[:] = Vn                              # copy, do NOT rebind: V is Vn otherwise
        outer += 1
        for _ in range(n_howard):              # Howard: same rhs at a frozen policy
            EV = V @ model.Pi.T
            for j in range(model.n_z):
                V[:, j] = rhs(g[:, j], model.y[:, j], EV[:, j], model, kg)
        if d < tol: break
    return V, g, outer, time.perf_counter() - t0, nev

`V = Vn` would **rebind**, not copy: the Howard loop would then write into the array the
convergence test reads, `d` would be identically zero, and the solver would stop after one
sweep with a wrong answer. `V[:] = Vn` is the fix. This is a bug worth writing once on
purpose so you recognise it later.

In [ ]:
_, g_grid, it_g, t_g = vfi(m, n_howard=0)              # the crude Session 3 solver
_, g_cont, it_c, t_c, nev = vfi_continuous(m)

print(f"grid search : {it_g:5d} sweeps          {t_g:6.2f} s   "
      f"{m.n_k*m.n_k*m.n_z*it_g:>14,} comparisons")
print(f"golden sect.: {it_c:5d} policy updates  {t_c:6.2f} s   {nev:>14,} evaluations")

Two things to say out loud about that table. The **iteration count** fell because of Howard,
not because of golden section. And the **wall time went up** even though the inner work fell
by a large factor, because golden section runs in a Python loop while the grid search is one
array operation. Report both; a speed claim without a timing is not a result.

In [ ]:
# Does the continuous policy actually behave? Three checks, in this order.
jm = m.n_z // 2
print("1. ordered in z at the middle of the grid:",
      np.all(np.diff(g_cont[m.n_k//2, :]) > 0))
print("2. increasing in k in every shock state: ",
      np.all(np.diff(g_cont, axis=0) > 0))
sav = g_cont[:, jm] - m.k
cross = m.k[np.argmin(np.abs(sav))]
print(f"3. middle-state 45-degree crossing at {cross:.4f}, analytic k* = {m.kss:.4f} "
      f"({abs(cross-m.kss)/(m.k[1]-m.k[0]):.1f} grid spacings)")

In [ ]:
# The picture that makes the difference visible: plot SAVING, not the policy.
# g(k) against k is three overlapping straight lines; g(k) - k is not.
sav_g, sav_c = g_grid[:, jm] - m.k, g_cont[:, jm] - m.k
w = slice(120, 156)                                   # 36 consecutive grid points

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4))
for j, lab in ((0, 'z low'), (jm, 'z middle'), (m.n_z-1, 'z high')):
    ax[0].plot(m.k, g_cont[:, j] - m.k, lw=1.3, label=lab)
ax[0].axhline(0, color='grey', ls='--', lw=0.8)
ax[0].axvline(m.kss, color='grey', ls=':', lw=0.8)
ax[0].set_xlabel("$k$"); ax[0].set_ylabel("saving $g(k,z)-k$")
ax[0].legend(frameon=False); ax[0].grid(alpha=0.3)

ax[1].plot(m.k[w], sav_g[w], 's-', ms=3, color='grey', lw=1, label='grid search')
ax[1].plot(m.k[w], sav_c[w], 'o-', ms=2.5, color='C3', lw=1.3, label='golden section')
ax[1].axhline(0, color='grey', ls='--', lw=0.8)
ax[1].set_xlabel("$k$ (36 consecutive grid points)"); ax[1].set_ylabel("saving")
ax[1].legend(frameon=False); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("distinct policy values in those 36 states: "
      f"grid search {len(np.unique(g_grid[w, jm]))}, "
      f"golden section {len(np.unique(g_cont[w, jm]))}")

The right-hand panel is the whole argument in one picture. Grid search can only choose
$k' = k + mh$, so in this window **saving takes two values**: one grid spacing, then zero. Its
zero-crossing lands about ten grid spacings before the analytic $k^*$. Golden section gives a
smooth declining line that crosses once, in the right place.

Plotted as $k'$ against $k$ instead, the two are indistinguishable — the differences are
$O(h)$ on an axis that spans 28 units. **Subtract the benchmark before you plot.**

**Exercise 7.** Compute off-grid Euler residuals for both policies. The lecture measured
$10^{-1.30}$ against $10^{-2.52}$ at $n_k=300$ — about 1.2 decades, i.e. the error falls by a
factor of $10^{1.2} \approx 16$. Do you reproduce it? Then rerun at $n_k = 500$ and check
that *both* improve while the gap stays near 1.2 decades.

**Exercise 8.** Replace `np.interp` with `CubicSpline` inside `rhs` and re-run. The answer may
still look fine. Test the thing that actually broke: check whether $\hat v$ is concave at
every iteration, and report how often it is not.

---
## What to take away

1. Convergence orders are real and measurable: 1, 1.618, 2. Bracketing methods are the ones
   that tell you when you have given them an impossible problem.
2. Golden section costs $O(\log(1/arepsilon))$ and is **independent of the state grid**.
3. Accuracy order is not the criterion for interpolation. **Shape is.**
4. A quadrature rule that knows the weight function beats one that does not, by decades.
5. **Rouwenhorst, not Tauchen**, whenever $ho$ is near one — which in macro is always.
6. The largest single gain in this lab came from **reformulating the state**, not from a
   better algorithm.
7. Interpolation is not a display choice inside a Bellman loop; it is a **hypothesis of the
   maximizer**. Linear keeps concavity, concavity gives unimodality, unimodality is what
   licenses golden section.
8. **Plot the deviation, not the level.** Differences between solvers are $O(h)$; a level plot
   on a wide axis hides them completely.